### Bước 1: Kiểm tra môi trường & Cài đặt thư viện bổ sung

 sẽ kiểm tra xem Colab có đang chạy trên môi trường hỗ trợ GPU hay không, kiểm tra phiên bản PyTorch và thư viện Transformers của Hugging Face, đồng thời cài đặt thư viện `seqeval` phục vụ cho việc đánh giá mô hình NER sau này.

In [6]:
!pip install -q seqeval transformers[torch] datasets unicodedata2

In [7]:
import torch
import transformers

print(f"CUDA (GPU) available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.version.__version__}")
print(f"Transformers version: {transformers.__version__}")

CUDA (GPU) available: True
GPU Name: Tesla T4
PyTorch version: 2.14.1+cu130
Transformers version: 5.22.1


### Bước 2: Đọc dữ liệu, Chuẩn hóa Unicode NFC & Chuẩn hóa nhãn
Dữ liệu gốc từ file `/content/Vn_resumes_NER.txt` thường ở định dạng gán nhãn từng từ CoNLL (từ và nhãn cách nhau bởi khoảng trắng):
- Đọc file và gom cụm thành các câu/đoạn văn hoàn chỉnh.
- Chuẩn hóa text sang Unicode NFC sử dụng thư viện `unicodedata`.
- Chuyển đổi nhãn từ định dạng BIO ban đầu (O, B-SKILL, I-SKILL) thành hệ nhãn đơn giản nhị phân: `O` và `SKILL` (gộp B-SKILL và I-SKILL).

In [8]:
import os
import unicodedata

def load_and_preprocess_data(file_path):
    sentences = []
    current_words = []
    current_labels = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_words:
                    sentences.append((current_words, current_labels))
                    current_words = []
                    current_labels = []
                continue

            parts = line.split()
            if len(parts) >= 2:
                word = " ".join(parts[:-1])
                label = parts[-1]

                # Chuẩn hóa Unicode NFC
                word = unicodedata.normalize('NFC', word)

                # Chuẩn hóa nhãn thành nhị phân: O hoặc SKILL
                if label in ['B-SKILL', 'I-SKILL', 'SKILL']:
                    mapped_label = 'SKILL'
                else:
                    mapped_label = 'O'

                current_words.append(word)
                current_labels.append(mapped_label)

        if current_words:
            sentences.append((current_words, current_labels))

    return sentences

data_path = '/content/Vn_resumes_NER.txt'
all_data = load_and_preprocess_data(data_path)
print(f"Tổng số câu/mẫu dữ liệu đã đọc: {len(all_data)}")

Tổng số câu/mẫu dữ liệu đã đọc: 19323


### Bước 3: Phân tích phân bổ nhãn để phát hiện sự mất cân bằng (Class Imbalance)
 thống kê tỷ lệ của nhãn `SKILL` so với nhãn `O` trên toàn bộ tập dữ liệu để có cái nhìn tổng quan về phân bố nhãn.

In [9]:
from collections import Counter
import pandas as pd

all_labels = [label for _, labels in all_data for label in labels]
label_counts = Counter(all_labels)

df_dist = pd.DataFrame.from_dict(label_counts, orient='index', columns=['Tần suất'])
df_dist['Tỷ lệ (%)'] = (df_dist['Tần suất'] / df_dist['Tần suất'].sum()) * 100
display(df_dist)

,Tần suất,Tỷ lệ (%)
O,239984,76.544305
SKILL,73539,23.455695


### Bước 4: Chia tập dữ liệu (80% Train, 10% Val, 10% Test) & Lưu trữ dưới dạng JSON
Sử dụng thư viện `scikit-learn` hoặc chia ngẫu nhiên thủ công một cách đồng nhất, sau đó xuất kết quả ra các file JSON tương ứng để nạp nhanh chóng ở các pha huấn luyện sau.

In [10]:
import random
import json

# Thiết lập seed để đảm bảo kết quả phân chia có thể tái lập
random.seed(42)
random.shuffle(all_data)

total_samples = len(all_data)
train_idx = int(0.8 * total_samples)
val_idx = train_idx + int(0.1 * total_samples)

train_data = all_data[:train_idx]
val_data = all_data[train_idx:val_idx]
test_data = all_data[val_idx:]

print(f"Tập Train: {len(train_data)} mẫu")
print(f"Tập Validation: {len(val_data)} mẫu")
print(f"Tập Test: {len(test_data)} mẫu")

def save_to_json(data, output_path):
    formatted_data = []
    for words, labels in data:
        formatted_data.append({
            "tokens": words,
            "ner_tags": labels
        })
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(formatted_data, f, ensure_ascii=False, indent=4)

save_to_json(train_data, 'train.json')
save_to_json(val_data, 'val.json')
save_to_json(test_data, 'test.json')
print("Đã lưu thành công các tệp: train.json, val.json, test.json.")

Tập Train: 15458 mẫu
Tập Validation: 1932 mẫu
Tập Test: 1933 mẫu
Đã lưu thành công các tệp: train.json, val.json, test.json.


### Giai đoạn 2: Tokenize & Thiết lập PyTorch Pipeline

#### Bước 2.1: Khởi tạo PhoBERT Tokenizer
 bộ Tokenizer của mô hình `vinai/phobert-base`. Do PhoBERT yêu cầu văn bản đã được phân tách từ (segmented word-level), nên dữ liệu từ các dòng trong sơ yếu lý lịch đã được tách từ sẵn theo khoảng trắng trong file raw là rất phù hợp.

In [11]:
from transformers import AutoTokenizer

# Sử dụng mô hình PhoBERT-base
model_checkpoint = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False)

print(f"PhoBERT Tokenizer loaded. Vocabulary size: {len(tokenizer)}")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

PhoBERT Tokenizer loaded. Vocabulary size: 64001


#### Bước 2.2: Căn chỉnh nhãn (Label Alignment) cho Sub-tokens
Vì một từ phức hoặc một từ đơn có thể bị chia tách thành nhiều sub-tokens (ví dụ: `Python` có thể giữ nguyên, nhưng một số từ tiếng Việt lạ có thể bị phân rã), chúng ta cần gán nhãn thực tế cho token đầu tiên của từ đó, và gán nhãn `-100` cho các token con tiếp theo cũng như các token đặc biệt (`<s>`, `</s>`).

In [12]:
# Định nghĩa ánh xạ nhãn chữ sang số nguyên
label2id = {"O": 0, "SKILL": 1}
id2label = {0: "O", 1: "SKILL"}

def align_labels_with_tokens(tokens, labels, tokenizer):
    # Sử dụng tokenizer để mã hóa câu
    tokenized_input = tokenizer(
        tokens,
        is_split_into_words=True,
        return_offsets_mapping=False, # PhoBERT cũ (không phải fast) không hỗ trợ offset_mapping trực tiếp dễ dàng
        truncation=True,
        max_length=256
    )

    aligned_labels = []
    # Để căn chỉnh thủ công với tokenizer không phải Fast, ta duyệt qua các token
    # PhoBERT tokenizer.tokenize() trả về các sub-tokens kèm tiền tố '@@' nếu không phải từ kết thúc

    word_idx = -1
    sub_tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])

    for token in sub_tokens:
        if token in [tokenizer.bos_token, tokenizer.eos_token, tokenizer.pad_token]:
            aligned_labels.append(-100)
        else:
            # Kiểm tra xem sub-token này có phải phần tiếp theo của từ trước hay không
            # Đối với PhoBERT (BPE), các token trung gian thường kết thúc bằng '@@'
            # Nếu token trước đó có '@@', token hiện tại thuộc cùng một từ gốc
            if word_idx >= 0 and sub_tokens[sub_tokens.index(token)-1].endswith('@@'):
                aligned_labels.append(-100)
            else:
                word_idx += 1
                if word_idx < len(labels):
                    aligned_labels.append(label2id[labels[word_idx]])
                else:
                    aligned_labels.append(-100)

    tokenized_input["labels"] = aligned_labels
    return tokenized_input

#### Bước 2.3: Xây dựng cấu trúc PyTorch Dataset và DataLoader với Dynamic Padding

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForTokenClassification

class ResumeNERDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item["tokens"]
        labels = item["ner_tags"]

        aligned_dict = align_labels_with_tokens(tokens, labels, self.tokenizer)

        # Chuyển đổi sang Tensor PyTorch
        return {
            "input_ids": torch.tensor(aligned_dict["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(aligned_dict["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(aligned_dict["labels"], dtype=torch.long)
        }

# Khởi tạo các Dataset
train_dataset = ResumeNERDataset(train_data, tokenizer)
val_dataset = ResumeNERDataset(val_data, tokenizer)
test_dataset = ResumeNERDataset(test_data, tokenizer)

# Sử dụng DataCollatorForTokenClassification để tự động pad theo độ dài lớn nhất trong batch (Dynamic Padding)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=data_collator)

print(f"Xây dựng Dataset & DataLoader thành công!")

Xây dựng Dataset & DataLoader thành công!


#### Bước 2.4: Tính toán Trọng số Nhãn (Class Weights)
Sử dụng công thức nghịch đảo tần suất mẫu để tính toán trọng số lỗi cho nhãn `O` và `SKILL`, giúp cải thiện hiệu năng cho lớp thiểu số `SKILL`.

In [14]:
import numpy as np

# Thống kê tần suất từ bước 1
total_tags = df_dist['Tần suất'].sum()
count_O = df_dist.loc['O', 'Tần suất']
count_SKILL = df_dist.loc['SKILL', 'Tần suất']

# Tính trọng số nghịch đảo
weight_O = total_tags / (2.0 * count_O)
weight_SKILL = total_tags / (2.0 * count_SKILL)

class_weights = torch.tensor([weight_O, weight_SKILL], dtype=torch.float)
print(f"Trọng số phạt lỗi (Class Weights) - Lớp O: {weight_O:.4f}, Lớp SKILL: {weight_SKILL:.4f}")

Trọng số phạt lỗi (Class Weights) - Lớp O: 0.6532, Lớp SKILL: 2.1317


### Giai đoạn 3: Huấn luyện & Đánh giá mô hình (Training & Evaluation)

#### Bước 3.1: Khởi tạo mô hình PhoBERT cho Token Classification
 nạp mô hình pre-trained `vinai/phobert-base` và cấu hình đầu ra cho 2 nhãn phân loại: `O` (id 0) và `SKILL` (id 1).

In [15]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Bước 3.2: Tùy biến Custom Trainer áp dụng Class Weights
Do dữ liệu mất cân bằng nghiêm trọng, nên ghi đè phương thức `compute_loss` của Hugging Face `Trainer` để tích hợp `class_weights` vào `CrossEntropyLoss`. Nhãn `-100` sẽ tự động bị bỏ qua trong tính toán loss nhờ thuộc tính `ignore_index`.

In [16]:
import torch.nn as nn
from transformers import Trainer

# Đảm bảo class_weights được chuyển tới thiết bị phù hợp (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = class_weights.to(device)

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Khởi tạo loss function với trọng số lớp
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor, ignore_index=-100)

        # Tính toán loss
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

#### Bước 3.3: Định nghĩa hàm Compute Metrics (Sử dụng Seqeval)
 sử dụng `seqeval` để đánh giá chất lượng mô hình dựa trên các nhãn chuỗi thực thể thực tế (bỏ qua các giá trị `-100`).

In [17]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Chuyển đổi ID dự đoán và ID nhãn thực tế về nhãn chuỗi chữ, lọc bỏ nhãn -100
    true_predictions = [
        [id2label[p_val] for (p_val, l_val) in zip(prediction, label) if l_val != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l_val] for (p_val, l_val) in zip(prediction, label) if l_val != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions)
    }

#### Bước 3.4: Thiết lập tham số và Tiến hành Huấn luyện
 tham số tối ưu và khởi chạy `CustomTrainer`.

In [18]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForTokenClassification, TrainingArguments

# 1. Định nghĩa lại Dataset sửa lỗi index tuple
class ResumeNERDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item[0]   # Sửa đổi từ item["tokens"] thành item[0]
        labels = item[1]   # Sửa đổi từ item["ner_tags"] thành item[1]

        aligned_dict = align_labels_with_tokens(tokens, labels, self.tokenizer)

        return {
            "input_ids": torch.tensor(aligned_dict["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(aligned_dict["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(aligned_dict["labels"], dtype=torch.long)
        }

# Khởi tạo lại các Dataset chính xác
train_dataset = ResumeNERDataset(train_data, tokenizer)
val_dataset = ResumeNERDataset(val_data, tokenizer)
test_dataset = ResumeNERDataset(test_data, tokenizer)

# 2. Khởi tạo cấu hình và bộ huấn luyện
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# Tiến hành huấn luyện mô hình
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.172969,0.164111,0.822446,0.938068,0.876461,0.935762
2,0.135825,0.165606,0.841633,0.929106,0.883209,0.940311
3,0.111602,0.163435,0.834550,0.933119,0.881086,0.938816


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2901, training_loss=0.1554626342074824, metrics={'train_runtime': 354.0121, 'train_samples_per_second': 130.996, 'train_steps_per_second': 8.195, 'total_flos': 1544808881013264.0, 'train_loss': 0.1554626342074824, 'epoch': 3.0})

#### Bước 3.5: Đánh giá chi tiết trên tập Test độc lập & Lưu trữ
Sau khi huấn luyện hoàn tất, chúng ta dự đoán trên tập dữ liệu kiểm thử độc lập (Test Set), hiển thị Classification Report và lưu mô hình tốt nhất vào thư mục `best_phobert_ner`.

In [19]:
# Dự đoán trên tập Test
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

# Chuyển đổi nhãn về định dạng văn bản để tính báo cáo chi tiết
true_predictions = [
    [id2label[p_val] for (p_val, l_val) in zip(prediction, label) if l_val != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [id2label[l_val] for (p_val, l_val) in zip(prediction, label) if l_val != -100]
    for prediction, label in zip(predictions, labels)
]

# Xuất báo cáo hiệu năng phân loại thực thể chi tiết
report = classification_report(true_labels, true_predictions)
print("=== BÁO CÁO HIỆU NĂNG TRÊN TẬP KIỂM THỬ (TEST SET) ===")
print(report)

# Đóng gói và lưu trữ mô hình cùng tokenizer tối ưu
save_directory = "./best_phobert_ner"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Đã lưu mô hình tối ưu đạt F1-score cao nhất vào thư mục: {save_directory}")

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


=== BÁO CÁO HIỆU NĂNG TRÊN TẬP KIỂM THỬ (TEST SET) ===
              precision    recall  f1-score   support

        KILL       0.84      0.93      0.89      7223

   micro avg       0.84      0.93      0.89      7223
   macro avg       0.84      0.93      0.89      7223
weighted avg       0.84      0.93      0.89      7223



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình tối ưu đạt F1-score cao nhất vào thư mục: ./best_phobert_ner


In [20]:
# KIỂM CHỨNG BƯỚC CHUẨN HÓA TỪ ĐỒNG NGHĨA / BIẾN THỂ KỸ NĂNG
# Cell này không huấn luyện lại mô hình. Nó kiểm tra mô hình đang phản ánh điều gì:
# cùng một kỹ năng được viết dưới nhiều biến thể có được nhận diện nhất quán hay không.
# Lưu ý: alias map là một baseline minh bạch, không phải bằng chứng mô hình tự hiểu đồng nghĩa.

import re
import unicodedata
import pandas as pd
import torch


def _alias_key(text):
    text = unicodedata.normalize("NFC", str(text)).lower().strip()
    text = text.replace("_", " ")
    return re.sub(r"\s+", " ", text)


# Từ điển kiểm chứng nhỏ, có thể mở rộng bằng dữ liệu đã được chuyên gia duyệt.
# Các mục dưới đây biểu diễn biến thể/viết tắt trong CV, không khẳng định mọi ngữ cảnh
# đều là đồng nghĩa tuyệt đối.
SKILL_ALIAS_GROUPS = {
    "python": ["python", "py", "python3"],
    "javascript": ["javascript", "js"],
    "sql server": ["sql_server", "mssql", "ms_sql_server"],
    "làm việc nhóm": ["teamwork", "làm_việc_nhóm"],
    "giao tiếp": ["giao_tiếp", "communication"],
}
ALIAS_TO_CANONICAL = {
    _alias_key(alias): canonical
    for canonical, aliases in SKILL_ALIAS_GROUPS.items()
    for alias in aliases
}


def predict_word_level(sentence, threshold=0.50):
    """Dự đoán SKILL/O ở mức từ bằng mô hình đã huấn luyện trong các cell trước."""
    words = sentence.split()
    inference_model = trainer.model if "trainer" in globals() else model
    inference_device = next(inference_model.parameters()).device
    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    )
    encoded = {k: v.to(inference_device) for k, v in encoded.items()}
    inference_model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(inference_model(**encoded).logits[0], dim=-1)

    # PhoBERT/RoBERTa dùng 2 token đặc biệt: <s> và </s>.
    cursor = 1
    rows = []
    for word in words:
        piece_count = max(1, len(tokenizer.tokenize(word)))
        piece_probs = probabilities[cursor:cursor + piece_count]
        if piece_probs.shape[0] == 0:
            break
        skill_probability = float(piece_probs[:, label2id["SKILL"]].max().item())
        rows.append({
            "token": word,
            "label": "SKILL" if skill_probability >= threshold else "O",
            "skill_probability": skill_probability,
            "canonical": ALIAS_TO_CANONICAL.get(_alias_key(word)),
        })
        cursor += piece_count
    return rows


probe_rows = []
for canonical, variants in SKILL_ALIAS_GROUPS.items():
    for variant in variants:
        sentence = f"ứng_viên có kinh_nghiệm với {variant}"
        prediction = predict_word_level(sentence)
        target = next((row for row in prediction if _alias_key(row["token"]) == _alias_key(variant)), None)
        if target is None:
            raise ValueError(f"Không tìm thấy token kiểm chứng: {variant}")
        probe_rows.append({
            "canonical_skill": canonical,
            "variant": variant,
            "model_label": target["label"],
            "skill_probability": round(target["skill_probability"], 4),
            "dictionary_normalized": target["canonical"],
            # Baseline hợp nhất: giữ lại dự đoán của mô hình và cho phép từ điển
            # cứu các alias đã được chuyên gia xác nhận.
            "model_plus_alias_label": (
                "SKILL" if target["label"] == "SKILL" or target["canonical"] else "O"
            ),
        })

verification_df = pd.DataFrame(probe_rows)
group_summary = verification_df.groupby("canonical_skill").agg(
    variants=("variant", "count"),
    model_skill_rate=("model_label", lambda s: (s == "SKILL").mean()),
    alias_assisted_skill_rate=("model_plus_alias_label", lambda s: (s == "SKILL").mean()),
    min_probability=("skill_probability", "min"),
    max_probability=("skill_probability", "max"),
).reset_index()
group_summary["consistent_model_decision"] = group_summary["model_skill_rate"].isin([0.0, 1.0])

model_skill_rate = (verification_df["model_label"] == "SKILL").mean()
alias_assisted_skill_rate = (verification_df["model_plus_alias_label"] == "SKILL").mean()
rescued = ((verification_df["model_label"] == "O") &
           (verification_df["model_plus_alias_label"] == "SKILL")).sum()

print("=== KIỂM CHỨNG BIẾN THỂ KỸ NĂNG ===")
print("Mô hình NER tự động dự đoán SKILL/O; cell này không dùng prompt.")
print(f"Tỷ lệ biến thể được mô hình nhận diện là SKILL: {model_skill_rate:.2%}")
print(f"Tỷ lệ sau khi thêm alias map đã duyệt: {alias_assisted_skill_rate:.2%}")
print(f"Số trường hợp alias map cứu được từ dự đoán O: {rescued}")
print("\nChi tiết từng biến thể:")
display(verification_df)
print("\nTóm tắt theo kỹ năng chuẩn:")
display(group_summary)
print(
"\nDiễn giải: nếu model_skill_rate thấp hoặc quyết định không nhất quán trong cùng một nhóm, "
"mô hình chưa học được tính tương đương của các biến thể. Alias map chỉ là lớp chuẩn hóa "
"minh bạch để kiểm chứng và cần được đánh giá thêm về độ chính xác/ngữ cảnh trước khi đưa vào hệ thống."
)


=== KIỂM CHỨNG BIẾN THỂ KỸ NĂNG ===
Mô hình NER tự động dự đoán SKILL/O; cell này không dùng prompt.
Tỷ lệ biến thể được mô hình nhận diện là SKILL: 100.00%
Tỷ lệ sau khi thêm alias map đã duyệt: 100.00%
Số trường hợp alias map cứu được từ dự đoán O: 0

Chi tiết từng biến thể:


,canonical_skill,variant,model_label,skill_probability,dictionary_normalized,model_plus_alias_label
0,python,python,SKILL,0.9977,python,SKILL
1,python,py,SKILL,0.9972,python,SKILL
2,python,python3,SKILL,0.9971,python,SKILL
3,javascript,javascript,SKILL,0.9980,javascript,SKILL
4,javascript,js,SKILL,0.9979,javascript,SKILL
5,sql server,sql_server,SKILL,0.9975,sql server,SKILL
6,sql server,mssql,SKILL,0.9970,sql server,SKILL
7,sql server,ms_sql_server,SKILL,0.9978,sql server,SKILL
8,làm việc nhóm,teamwork,SKILL,0.9975,làm việc nhóm,SKILL
9,làm việc nhóm,làm_việc_nhóm,SKILL,0.9937,làm việc nhóm,SKILL



Tóm tắt theo kỹ năng chuẩn:


,canonical_skill,variants,model_skill_rate,alias_assisted_skill_rate,min_probability,max_probability,consistent_model_decision
0,giao tiếp,2,1.0,1.0,0.9935,0.9984,True
1,javascript,2,1.0,1.0,0.9979,0.9980,True
2,làm việc nhóm,2,1.0,1.0,0.9937,0.9975,True
3,python,3,1.0,1.0,0.9971,0.9977,True
4,sql server,3,1.0,1.0,0.9970,0.9978,True



Diễn giải: nếu model_skill_rate thấp hoặc quyết định không nhất quán trong cùng một nhóm, mô hình chưa học được tính tương đương của các biến thể. Alias map chỉ là lớp chuẩn hóa minh bạch để kiểm chứng và cần được đánh giá thêm về độ chính xác/ngữ cảnh trước khi đưa vào hệ thống.
